In [11]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import numpy as np
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.fit_params.weibull_fit_params import WeibullFitParams

plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

In [12]:
power_law_db = pl.read_parquet(r"MLE_random_sample_fit_data\PowerLaw")
weibull_db = pl.read_parquet(r"MLE_random_sample_fit_data\Weibull")
log_normal_db = pl.read_parquet(r"MLE_random_sample_fit_data\LogNormal")

In [13]:
log_normal_db

seed,aic,bic,numb_alphas,s_max_fitting_alpha,s_min_fitting_alpha,mu,sigma,mu_err,sigma_err,LAD_min,J_min,min_alpha_to_consider
i32,f64,f64,i64,f64,f64,f64,f64,f64,f64,i32,f64,i32
34791,1.8501e6,1.8501e6,129363,8616.625643,22.776133,-4.972953,2.268432,0.126724,0.027809,0,0.7,155
12582,1.8483e6,1.8483e6,129363,18217.221331,22.776133,-4.728107,2.222093,0.117545,0.026505,0,0.7,155
89725,1.8515e6,1.8515e6,129363,34166.764882,22.776133,-4.058933,2.083194,0.091073,0.022286,0,0.7,155
72097,1.8466e6,1.8466e6,129363,20845.303305,23.497827,-5.57859,2.394971,0.155237,0.032003,0,0.7,155
71723,1.8497e6,1.8498e6,129363,34166.764882,22.776133,-8.479012,2.85337,0.308514,0.0506,0,0.7,155
…,…,…,…,…,…,…,…,…,…,…,…,…
23122,605739.792867,605773.899218,37294,18217.221331,22.776133,-3.03323,1.972994,0.209661,0.047735,0,0.7,605
69050,605516.04061,605550.146961,37294,18217.221331,22.776133,-3.400343,2.049105,0.242551,0.052823,0,0.7,605
98961,605339.267819,605373.37417,37294,8541.69122,22.776133,-3.951867,2.150868,0.291493,0.059601,0,0.7,605


In [14]:
pl_df = power_law_db.group_by("min_alpha_to_consider").agg(
    (- 2 * pl.col("q")).mean().alias("b_mu"),
    (- 2 * pl.col("q")).std().alias("b_std")
).sort("min_alpha_to_consider")

w_df = weibull_db.group_by("min_alpha_to_consider").agg(
    pl.col("lambda").mean().alias("lambda_mu"),
    pl.col("lambda").std().alias("lambda_std"),
    pl.col("k").mean().alias("k_mu"),
    pl.col("k").std().alias("k_std")
).filter(pl.col("min_alpha_to_consider") < 1e4).sort("min_alpha_to_consider")

ln_df = log_normal_db.group_by("min_alpha_to_consider").agg(
    pl.col("mu").mean().alias("mu_mu"),
    pl.col("mu").std().alias("mu_std"),
    pl.col("sigma").mean().alias("sigma_mu"),
    pl.col("sigma").std().alias("sigma_std")
).filter(pl.col("mu_mu") > -200).sort("min_alpha_to_consider")

In [15]:
import pymannkendall as mk

# === PL === 

result = mk.original_test(
    pl_df.sort("min_alpha_to_consider")[f"b_mu"]
)
print(result)

# === W ===

result = mk.original_test(
    w_df.sort("min_alpha_to_consider")[f"lambda_mu"]
)
print(result)

result = mk.original_test(
    w_df.sort("min_alpha_to_consider")[f"k_mu"]
)
print(result)

# === LN ===

result = mk.original_test(
    ln_df.sort("min_alpha_to_consider")[f"mu_mu"]
)
print(result)

result = mk.original_test(
    ln_df.sort("min_alpha_to_consider")[f"sigma_mu"]
)
print(result)

Mann_Kendall_Test(trend='decreasing', h=np.True_, p=np.float64(1.8187150270243535e-06), z=np.float64(-4.772590320194992), Tau=np.float64(-0.8300653594771242), s=np.float64(-127.0), var_s=697.0, slope=np.float64(-0.04858547826359216), intercept=np.float64(-2.102110088934407))
Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.2263989227367953), z=np.float64(-1.2096872536109093), Tau=np.float64(-0.13630229419703105), s=np.float64(-101.0), var_s=6833.666666666667, slope=np.float64(-5.843204887262196e-16), intercept=np.float64(5.6656200278379065e-14))
Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.2554939683022921), z=np.float64(-1.1371060183942547), Tau=np.float64(-0.1282051282051282), s=np.float64(-95.0), var_s=6833.666666666667, slope=np.float64(-0.0001359371053362516), intercept=np.float64(0.08752294655911841))
Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.7425568219265775), z=np.float64(-0.32846934087081536), Tau=np.float64(-0.0769230769

In [16]:
plot_param  = lambda xs, ys, yerrs, c, symbol : plt.plot(
    xs, (ys - ys.mean()) / ys.std(), color = c, label = rf"${symbol} - \bar{{{symbol}}}/{symbol}_{{min}}$"
)

# plt.errorbar(
#     xs,
#     (ys - ys.mean())/ ys.std(),
#     yerr = yerrs / ys.std(),
#     marker=".",
#     capsize=2,
#     linestyle="none",
#     ms = 3,
#     color = c,
#     label = f"${symbol}/{symbol}_{{min}}$"
# )

plot_param(pl_df["min_alpha_to_consider"], pl_df["q_mu"], pl_df["q_std"], "C0", "q")

plot_param(w_df["min_alpha_to_consider"], w_df["lambda_mu"], w_df["lambda_std"], "C1", r"\lambda")
plot_param(w_df["min_alpha_to_consider"], w_df["k_mu"], w_df["k_std"], "C2", r"k")

plot_param(ln_df["min_alpha_to_consider"], ln_df["mu_mu"], ln_df["mu_std"], "C3", r"\mu")
plot_param(ln_df["min_alpha_to_consider"], ln_df["sigma_mu"], ln_df["sigma_std"], "C4", r"\sigma")

# plt.ylim([-1, 1])
plt.xscale("log")
plt.xlabel(r"$\alpha_{min}$")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

ColumnNotFoundError: "q_mu" not found